# 19 · Auditoría de afirmaciones del GraphRAG

Esta etapa **no vuelve a llamar a Groq**, no necesita OCR y no recalcula embeddings. Examina las respuestas de Qwen de la etapa 18 y diferencia:

1. **Cita identificable**: el MCID aparece entre las fuentes recuperadas.
2. **Evidencia catalográfica comprobada**: el fragmento está literalmente en un campo de la ficha.
3. **Afirmación históricamente respaldada**: **no** puede darse por cierta automáticamente; requiere lectura del documento y juicio humano.

Genera automáticamente `data/output/etapa19.zip`, con una matriz de afirmaciones y una plantilla de evaluación. Se conservan las respuestas originales, incluidas las no aprobadas, para mantener la trazabilidad.

**Uso:** abre este archivo dentro de `notebooks/` y pulsa **Run → Run All Cells**. El proyecto incluye `data/input/etapa18.zip`.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from io import BytesIO
import json, zipfile, re, hashlib, shutil
import pandas as pd

HERE=Path.cwd().resolve()
ROOT=next((p for p in (HERE,*HERE.parents) if (p/'notebooks').is_dir() and (p/'data').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('Abre Jupyter dentro del proyecto memoria_chilena (carpetas notebooks/ y data/).')
ENTRADA=ROOT/'data'/'input'/'etapa18.zip'
if not ENTRADA.exists():
    alternativa=ROOT/'data'/'output'/'etapa18.zip'
    if alternativa.exists(): ENTRADA=alternativa
if not ENTRADA.exists():
    raise FileNotFoundError('No encuentro etapa18.zip en data/input ni data/output.')
DEST=ROOT/'data'/'output'/'etapa19'
DEST.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(ENTRADA) as z:
    nombres=set(z.namelist())
    requeridos={'respuestas_groq.csv','fuentes_recuperadas.csv','evidencias_catalograficas.csv','informe_etapa18.json'}
    assert requeridos.issubset(nombres), f'Faltan archivos: {requeridos-nombres}'
    respuestas=pd.read_csv(BytesIO(z.read('respuestas_groq.csv')),dtype=str,keep_default_na=False,encoding='utf-8-sig')
    fuentes=pd.read_csv(BytesIO(z.read('fuentes_recuperadas.csv')),dtype=str,keep_default_na=False,encoding='utf-8-sig')
    evidencias=pd.read_csv(BytesIO(z.read('evidencias_catalograficas.csv')),dtype=str,keep_default_na=False,encoding='utf-8-sig')

assert len(respuestas)==5 and len(fuentes)==20, (len(respuestas),len(fuentes))
print('Entrada:',ENTRADA)
print('Respuestas:',len(respuestas),' | Fuentes:',len(fuentes),' | Evidencias:',len(evidencias))

Entrada: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa18.zip
Respuestas: 5  | Fuentes: 20  | Evidencias: 14


## 1. Auditoría de citas y evidencias

La comprobación es estrictamente determinista. Para cada respuesta se verifican los MCID citados, su pertenencia al contexto, la presencia de una evidencia y la coincidencia literal del fragmento con el campo catalográfico permitido. **Las afirmaciones no se marcan como verdaderas automáticamente.**

In [2]:
CAMPOS={'titulo','autor','fecha_catalografica','materias','unidades_tematicas','notas','coleccion','derechos'}
MCID_RE=re.compile(r'\[(MC\d{5,})\]')

def norm(s):
    return re.sub(r'\s+',' ',str(s or '')).strip().casefold()

def idcitas(texto):
    return sorted(set(MCID_RE.findall(str(texto))))

def divide_afirmaciones(texto):
    # Divisor práctico, no analizador semántico: preservar la respuesta completa.
    piezas=re.split(r'(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÜÑ¿])',str(texto).strip())
    return [x.strip() for x in piezas if x.strip()]

def evidencias_para(pregunta,mcid):
    rows=evidencias[(evidencias.pregunta==pregunta) & (evidencias.mcid==mcid)]
    original=fuentes[(fuentes.pregunta==pregunta) & (fuentes.mcid==mcid)]
    if original.empty: return [],['MCID no recuperado']
    ficha=original.iloc[0]
    validas=[];problemas=[]
    for _,e in rows.iterrows():
        campo=e['campo']; frag=e['fragmento']
        if campo not in CAMPOS:
            problemas.append(f'Campo fuera de lista: {campo}')
        elif not frag.strip() or norm(frag) not in norm(ficha[campo]):
            problemas.append(f'Fragmento no coincide en {campo}')
        elif e['ficha']!=ficha['ficha']:
            problemas.append(f'Enlace discordante para {mcid}')
        else:
            validas.append({'mcid':mcid,'campo':campo,'fragmento':frag,'ficha':e['ficha']})
    return validas,problemas

# Indicadores de riesgo, no pruebas de error histórico.
RIESGOS=[
 ('afirmacion_sobre_contenido', r'\b(trata[sn]?|documenta[n]?|demuestra[n]?|explica[n]?|describe[n]?|relata[n]?|aborda[n]?|se centra|contiene[n]?)\b'),
 ('conclusion_exclusiva', r'\b(solo|únicamente|ningun[ao]?|no contiene[n]?|las demás|todos|ninguna)\b'),
 ('fecha_a_interpretar', r'\b(catalogad[ao] en|publicad[ao] en|se publicó en)\b'),
 ('relacion_historica_inferida', r'\b(influyó|originó|provocó|debido a|demuestra que|causó|fue consecuencia)\b'),
]

def riesgos(texto):
    return [tipo for tipo,exp in RIESGOS if re.search(exp,texto,flags=re.IGNORECASE)]

filas=[]
for j,r in respuestas.reset_index(drop=True).iterrows():
    pregunta=r['pregunta']; disponibles=set(fuentes[fuentes.pregunta==pregunta].mcid)
    for k,af in enumerate(divide_afirmaciones(r['respuesta']),1):
        citas=idcitas(af)
        problemas=[]; ev=[]
        if not citas: problemas.append('sin_cita_en_afirmacion')
        for mcid in citas:
            if mcid not in disponibles:
                problemas.append(f'cita_fuera_del_contexto:{mcid}')
                continue
            validas,problemas_e=evidencias_para(pregunta,mcid)
            ev.extend(validas)
            if not validas: problemas.append(f'cita_sin_evidencia_literal:{mcid}')
            problemas.extend([f'{mcid}:{p}' for p in problemas_e])
        rs=riesgos(af)
        estado='cita_y_evidencia_literal' if citas and not problemas else 'requiere_comprobacion_de_citas'
        filas.append({
            'pregunta_id':f'Q{j+1:02}', 'pregunta':pregunta,
            'afirmacion_id':f'Q{j+1:02}-A{k:02}', 'afirmacion':af,
            'citas': '|'.join(citas), 'control_literal':estado,
            'n_evidencias_validas':len(ev),'evidencias_json':json.dumps(ev,ensure_ascii=False),
            'alertas_citas':' | '.join(problemas), 'alertas_inferencia':' | '.join(rs),
            'juicio_humano':'', 'notas_revision':'',
            'advertencia':'El fragmento valida un campo catalográfico, NO toda la afirmación.'
        })
matriz=pd.DataFrame(filas)
matriz.to_csv(DEST/'auditoria_afirmaciones.csv',index=False,encoding='utf-8-sig')
print('Afirmaciones:',len(matriz))
print(matriz.groupby('control_literal').size().to_string())
print('Afirmaciones con alertas de inferencia:',int((matriz.alertas_inferencia!='').sum()))
display(matriz[['afirmacion_id','citas','control_literal','alertas_citas','alertas_inferencia']])

Afirmaciones: 16
control_literal
cita_y_evidencia_literal          7
requiere_comprobacion_de_citas    9
Afirmaciones con alertas de inferencia: 7


,afirmacion_id,citas,control_literal,alertas_citas,alertas_inferencia
0,Q01-A01,MC0031311|MC0061013|MC0077873,cita_y_evidencia_literal,,afirmacion_sobre_contenido
1,Q01-A02,,requiere_comprobacion_de_citas,sin_cita_en_afirmacion,
2,Q01-A03,,requiere_comprobacion_de_citas,sin_cita_en_afirmacion,
3,Q02-A01,MC0000958,cita_y_evidencia_literal,,afirmacion_sobre_contenido
4,Q02-A02,,requiere_comprobacion_de_citas,sin_cita_en_afirmacion,afirmacion_sobre_contenido | fecha_a_interpretar
5,Q02-A03,,requiere_comprobacion_de_citas,sin_cita_en_afirmacion,afirmacion_sobre_contenido | conclusion_exclusiva
6,Q02-A04,,requiere_comprobacion_de_citas,sin_cita_en_afirmacion,afirmacion_sobre_contenido | conclusion_exclusiva
7,Q03-A01,MC0001859,cita_y_evidencia_literal,,
8,Q03-A02,MC0027682|MC0049519,cita_y_evidencia_literal,,
9,Q03-A03,MC0018305,cita_y_evidencia_literal,,


## 2. Revisión humana

El archivo `revision_historica.csv` ofrece una fila por afirmación. Completa `juicio_humano` con **0** (no sustentada), **1** (sustento parcial/indirecto) o **2** (bien sustentada), y anota la fuente consultada. No confundas una evidencia del título con comprobación del contenido de la obra.

In [3]:
revision=matriz[['pregunta_id','pregunta','afirmacion_id','afirmacion','citas','control_literal','alertas_citas','alertas_inferencia','juicio_humano','notas_revision']].copy()
revision.to_csv(DEST/'revision_historica.csv',index=False,encoding='utf-8-sig')

# Comprobación de cobertura de las citas, por respuesta completa.
resumen=[]
for _,r in respuestas.iterrows():
    pregunta=r['pregunta']; citas=idcitas(r['respuesta'])
    sin_ev=[]; problemas=[]
    for m in citas:
        validas,malas=evidencias_para(pregunta,m)
        if not validas: sin_ev.append(m)
        problemas.extend(malas)
    resumen.append({
       'pregunta':pregunta,
       'estado_etapa18':r['estado'],
       'citas_total':len(citas), 'citas_sin_evidencia': '|'.join(sin_ev),
       'respuesta_completa':r['finish_reason']=='stop',
       'revision_historica':'pendiente',
       'alertas_evidencia':' | '.join(problemas)
    })
resumen_df=pd.DataFrame(resumen)
resumen_df.to_csv(DEST/'revision_por_respuesta.csv',index=False,encoding='utf-8-sig')
display(resumen_df)

,pregunta,estado_etapa18,citas_total,citas_sin_evidencia,respuesta_completa,revision_historica,alertas_evidencia
0,¿Qué obras tratan el voto de las mujeres en Ch...,citas_y_fragmentos_validos_revision_historica_...,3,,True,pendiente,
1,¿Qué fuentes documentan los inicios del Instit...,citas_y_fragmentos_validos_revision_historica_...,1,,True,pendiente,
2,¿Qué libros permiten investigar las celebracio...,citas_y_fragmentos_validos_revision_historica_...,4,,True,pendiente,
3,¿Qué documentos coloniales tratan sobre la len...,citas_y_fragmentos_validos_revision_historica_...,2,,True,pendiente,
4,¿Qué fichas se relacionan con las crónicas del...,requiere_revision_de_citas_y_evidencia,4,MC0013341,True,pendiente,


## 3. Respuesta de respaldo del caso fallido

Para la pregunta de las crónicas del Perú se construye, **sin LLM**, una respuesta de catálogo que cita solo los registros que contienen explícitamente la unidad temática `cronistasperuanos`. No se afirma que sean del mismo género histórico.

In [4]:
objetivo='¿Qué fichas se relacionan con las crónicas del Perú?'
sub=fuentes[fuentes.pregunta==objetivo].copy()
seleccion=sub[sub.unidades_tematicas.str.contains('cronistasperuanos',case=False,regex=False,na=False)]
lineas=['# Consulta: crónicas del Perú','',
        'Según las **unidades temáticas del catálogo**, las siguientes fichas están vinculadas a `cronistasperuanos`:','']
for _,f in seleccion.iterrows():
    lineas.append(f'- [{f["titulo"]}]({f["ficha"]}) [{f["mcid"]}]')
lineas+=['', 'Esta es una asociación **curatorial**. No prueba que las tres obras pertenezcan al mismo género literario ni describe su contenido interno.',
         '', 'Fuente de información: campos `titulo`, `unidades_tematicas` y `ficha` de Memoria Chilena.']
respuesta_segura='\n'.join(lineas)
(DEST/'respuesta_catalografica_respaldo.md').write_text(respuesta_segura,encoding='utf-8')
print(respuesta_segura)

# Consulta: crónicas del Perú

Según las **unidades temáticas del catálogo**, las siguientes fichas están vinculadas a `cronistasperuanos`:

- [Parte primera de la Chrónica del Perú : que tracta la demarcación de sus provincias, la descripción de ellas : las fundacion de las nuevas ciudades : los ritos y costumbres de los indios y otras cosas extrañas ](https://www.memoriachilena.gob.cl/602/w3-article-8649.html) [MC0014754]
- [Primera parte de los commentarios reales, que tratan del origen de los Yncas, reyes que fueron del Peru de su idolatria, leyes, y govierno en paz y en guerra : de sus vidas y conquistas, y de todo lo que fue aq](https://www.memoriachilena.gob.cl/602/w3-article-8650.html) [MC0014755]
- [Extirpación de la idolatria del Piru](https://www.memoriachilena.gob.cl/602/w3-article-8655.html) [MC0014760]

Esta es una asociación **curatorial**. No prueba que las tres obras pertenezcan al mismo género literario ni describe su contenido interno.

Fuente de información: campos 

## 4. Exportar `etapa19.zip`

Se incluyen los archivos originales de etapa 18 **para auditoría**, más las matrices nuevas. Se excluyen secretos, PDF y otros archivos de usuario. El ZIP generado se guarda en `data/output/etapa19.zip`.

In [5]:
info={
  'etapa':19,
  'fecha_utc':datetime.now(timezone.utc).isoformat(),
  'fuente':'etapa18.zip',
  'respuestas':len(respuestas),
  'afirmaciones':len(matriz),
  'respuestas_control_citas_aprobado':int(respuestas['estado'].str.startswith('citas_y_fragmentos_validos').sum()),
  'afirmaciones_con_citas_y_evidencia_literal':int((matriz.control_literal=='cita_y_evidencia_literal').sum()),
  'afirmaciones_con_alertas_de_inferencia':int((matriz.alertas_inferencia!='').sum()),
  'control':'validación determinista de metadatos catalográficos; revisión histórica pendiente',
  'sin_llamadas_a_groq':True,
  'key_exportada':False,
}
(DEST/'informe_etapa19.json').write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding='utf-8')
reporte=[
 '# Informe de auditoría de afirmaciones — Etapa 19','',
 f'- Respuestas de Qwen revisadas: **{len(respuestas)}**',
 f'- Respuestas que pasan la verificación de citas de etapa 18: **{info["respuestas_control_citas_aprobado"]}/{len(respuestas)}**',
 f'- Afirmaciones extraídas para revisión: **{len(matriz)}**',
 f'- Afirmaciones con citas + al menos una evidencia literal por cita: **{info["afirmaciones_con_citas_y_evidencia_literal"]}**',
 f'- Afirmaciones marcadas por expresiones de inferencia para revisión: **{info["afirmaciones_con_alertas_de_inferencia"]}**',
 '',
 '**Advertencia:** no son porcentajes de exactitud histórica. El verificador solo comprueba campos y literales del catálogo.',
 '',
 '## Pendientes','',
 '1. Completar `revision_historica.csv` con juicios humanos por afirmación.',
 '2. Revisar el tratamiento de `Ley de Elecciones (Texto definitivo)` (1934): su presencia en una colección no demuestra el alcance de sufragio femenino que regula.',
 '3. Distinguir las categorías `cronistasperuanos` y los géneros históricos efectivos.',
 '4. Interpretar `fecha_catalografica` con cautela: no siempre es fecha de publicación; no afirmar “catalogado en” por su valor.',
 '5. No presentar las citas como soporte del contenido de libros no revisados.',
]
(DEST/'INFORME_ETAPA19.md').write_text('\n'.join(reporte)+'\n',encoding='utf-8')

# Reproducibilidad: incorporar solo archivos de datos conocidos de etapa18.
with zipfile.ZipFile(ENTRADA) as origen:
    for nombre in ('respuestas_groq.csv','fuentes_recuperadas.csv','evidencias_catalograficas.csv','informe_etapa18.json'):
        (DEST/nombre).write_bytes(origen.read(nombre))

salida=ROOT/'data'/'output'/'etapa19.zip'
with zipfile.ZipFile(salida,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DEST.iterdir()):
        if f.is_file(): z.write(f,arcname=f.name)
with zipfile.ZipFile(salida) as z:
    assert z.testzip() is None
    print('Archivos:', ', '.join(z.namelist()))
print('LISTO:',salida)

Archivos: auditoria_afirmaciones.csv, evidencias_catalograficas.csv, fuentes_recuperadas.csv, informe_etapa18.json, informe_etapa19.json, INFORME_ETAPA19.md, respuesta_catalografica_respaldo.md, respuestas_groq.csv, revision_historica.csv, revision_por_respuesta.csv
LISTO: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa19.zip
